# Nettoyage et préparation du dataset BIRDS - TSURU

Ce notebook prépare les données énergétiques réelles du CubeSat TSURU issues du programme BIRDS.

Objectifs :

- lire les différentes feuilles Excel ;
- reconstruire un vrai `measurement_datetime` ;
- calculer les puissances batterie et solaire ;
- récupérer la météo spatiale correspondante ;
- préparer les données pour SQL Server.

In [2]:
import pandas as pd
import numpy as np
import requests as rq

from datetime import datetime, timedelta

## 1. Paramètres du projet

Le fichier TSURU contient plusieurs feuilles correspondant à différentes sessions de mesure.

Le champ `Time stamp` n'est pas une date réelle.  
Il représente un temps écoulé depuis le début de la session.

On reconstruit donc un vrai timestamp avec :

`date de la feuille + Time stamp`

In [3]:
EXCEL_PATH = "TSURU.xlsx"

SATELLITE_NAME = "TSURU"
DEFAULT_YEAR = 2021

## 2. Lecture des feuilles Excel

In [4]:
xls = pd.ExcelFile(EXCEL_PATH)

xls.sheet_names[:10]

['March 28',
 'March 31',
 'April 20',
 'April 27',
 'May 06',
 'May 13',
 'May 21',
 'May 28',
 'June 01',
 'June 25']

## 3. Fonctions utiles

Cette fonction transforme le nom d'une feuille en date.

Exemple :

`March 28` devient `2021-03-28`.

Si une feuille contient déjà une année, elle sera utilisée.

In [5]:
def parse_sheet_date(sheet_name, default_year=2021):
    parts = sheet_name.strip().split()

    if len(parts) == 2:
        date_str = f"{parts[0]} {parts[1]} {default_year}"
    elif len(parts) == 3:
        date_str = f"{parts[0]} {parts[1]} {parts[2]}"
    else:
        return None

    try:
        return datetime.strptime(date_str, "%B %d %Y")
    except ValueError:
        return None

In [6]:
def to_number(series):
    return pd.to_numeric(series, errors="coerce")

## 4. Nettoyage du fichier TSURU

Pour chaque feuille :

- on lit les données ;
- on récupère la date à partir du nom de la feuille ;
- on reconstruit `measurement_datetime` ;
- on calcule la puissance batterie ;
- on calcule la puissance solaire totale à partir des 5 panneaux.

In [7]:
def clean_birds_excel(path, satellite_name="TSURU", default_year=2021):
    xls = pd.ExcelFile(path)
    all_data = []

    for sheet in xls.sheet_names:
        base_date = parse_sheet_date(sheet, default_year)

        if base_date is None:
            print(f"Feuille ignorée : {sheet}")
            continue

        df = pd.read_excel(path, sheet_name=sheet)

        if "Time stamp" not in df.columns:
            print(f"Pas de colonne Time stamp dans : {sheet}")
            continue

        df = df.copy()

        df["time_offset_s"] = to_number(df["Time stamp"])
        df = df.dropna(subset=["time_offset_s"])

        df["measurement_datetime"] = df["time_offset_s"].apply(
            lambda x: base_date + timedelta(seconds=float(x))
        )

        df["measurement_date"] = df["measurement_datetime"].dt.date
        df["measurement_time"] = df["measurement_datetime"].dt.time

        df["satellite_name"] = satellite_name
        df["sheet_name"] = sheet

        median_step = df["time_offset_s"].diff().median()

        if median_step <= 15:
            sampling_mode = "fast_sampling"
        else:
            sampling_mode = "nominal"

        df["sampling_mode"] = sampling_mode

        clean = pd.DataFrame({
            "satellite_name": df["satellite_name"],
            "sheet_name": df["sheet_name"],

            "measurement_datetime": df["measurement_datetime"],
            "measurement_date": df["measurement_date"],
            "measurement_time": df["measurement_time"],

            "time_offset_s": df["time_offset_s"],
            "sampling_mode": df["sampling_mode"],

            "battery_voltage_V": to_number(df["Vbat (V)"]),
            "battery_current_mA": to_number(df["Ibatt(mA)"]),
            "battery_temperature_C": to_number(df["Tbatt (℃)"]),

            "vpx_mV": to_number(df["Vpx (mV)"]),
            "vpy_mV": to_number(df["Vpy (mV)"]),
            "vpz_mV": to_number(df["Vpz (mV)"]),
            "vmx_mV": to_number(df["Vmx (mV)"]),
            "vmz_mV": to_number(df["Vmz (mV)"]),

            "ipx_mA": to_number(df["Ipx (mA)"]),
            "ipy_mA": to_number(df["Ipy (mA)"]),
            "ipz_mA": to_number(df["Ipz (mA)"]),
            "imx_mA": to_number(df["Imx (mA)"]),
            "imz_mA": to_number(df["Imz (mA)"]),

            "tpx_C": to_number(df["Tpx (°C)"]),
            "tpy_C": to_number(df["Tpy (°C)"]),
            "tpz_C": to_number(df["Tpz (°C)"]),
            "tmx_C": to_number(df["Tmx (°C)"]),
            "tmz_C": to_number(df["Tmz (°C)"]),
        })

        clean["battery_power_W"] = (
            clean["battery_voltage_V"]
            * clean["battery_current_mA"]
            / 1000
        )

        clean["solar_power_W"] = (
            clean["vpx_mV"] * clean["ipx_mA"]
            + clean["vpy_mV"] * clean["ipy_mA"]
            + clean["vpz_mV"] * clean["ipz_mA"]
            + clean["vmx_mV"] * clean["imx_mA"]
            + clean["vmz_mV"] * clean["imz_mA"]
        ) / 1_000_000

        all_data.append(clean)

    df_final = pd.concat(all_data, ignore_index=True)

    return df_final

In [8]:
df_birds = clean_birds_excel(
    EXCEL_PATH,
    satellite_name=SATELLITE_NAME,
    default_year=DEFAULT_YEAR
)

df_birds.head()

Feuille ignorée : Test1 w batt
Feuille ignorée : Test2 wo batt


,satellite_name,sheet_name,measurement_datetime,measurement_date,measurement_time,time_offset_s,sampling_mode,battery_voltage_V,battery_current_mA,battery_temperature_C,...,ipz_mA,imx_mA,imz_mA,tpx_C,tpy_C,tpz_C,tmx_C,tmz_C,battery_power_W,solar_power_W
0,TSURU,March 28,2021-03-28 00:00:00.000,2021-03-28,00:00:00,0.0,fast_sampling,4.1,280.91,10.26,...,9.32,0.0,4.66,6.19,4.63,8.29,-0.81,-3.91,1.151731,0.036493
1,TSURU,March 28,2021-03-28 00:00:11.300,2021-03-28,00:00:11.300000,11.3,fast_sampling,4.1,292.62,10.26,...,4.66,0.0,4.66,5.96,4.30,7.85,-1.25,-4.02,1.199742,0.030519
2,TSURU,March 28,2021-03-28 00:00:22.600,2021-03-28,00:00:22.600000,22.6,fast_sampling,4.1,351.19,10.26,...,4.66,0.0,4.66,5.41,3.63,7.30,-1.58,-4.36,1.439879,0.030519
3,TSURU,March 28,2021-03-28 00:00:33.900,2021-03-28,00:00:33.900000,33.9,fast_sampling,4.1,307.26,10.26,...,9.32,0.0,9.32,5.19,3.19,6.85,-2.03,-4.69,1.259766,0.042482
4,TSURU,March 28,2021-03-28 00:00:45.200,2021-03-28,00:00:45.200000,45.2,fast_sampling,4.1,257.48,10.26,...,4.66,0.0,4.66,4.97,2.63,6.52,-2.36,-5.02,1.055668,0.030512


## 5. Vérification des données nettoyées

In [9]:
df_birds.info()

<class 'pandas.DataFrame'>
RangeIndex: 23266 entries, 0 to 23265
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   satellite_name         23266 non-null  str           
 1   sheet_name             23266 non-null  str           
 2   measurement_datetime   23266 non-null  datetime64[us]
 3   measurement_date       23266 non-null  object        
 4   measurement_time       23266 non-null  object        
 5   time_offset_s          23266 non-null  float64       
 6   sampling_mode          23266 non-null  str           
 7   battery_voltage_V      23266 non-null  float64       
 8   battery_current_mA     23266 non-null  float64       
 9   battery_temperature_C  23266 non-null  float64       
 10  vpx_mV                 23266 non-null  float64       
 11  vpy_mV                 23266 non-null  float64       
 12  vpz_mV                 23266 non-null  float64       
 13  vmx_mV      

In [10]:
df_birds[
    [
        "measurement_datetime",
        "measurement_date",
        "measurement_time",
        "battery_voltage_V",
        "battery_current_mA",
        "battery_power_W",
        "solar_power_W"
    ]
].head()

,measurement_datetime,measurement_date,measurement_time,battery_voltage_V,battery_current_mA,battery_power_W,solar_power_W
0,2021-03-28 00:00:00.000,2021-03-28,00:00:00,4.1,280.91,1.151731,0.036493
1,2021-03-28 00:00:11.300,2021-03-28,00:00:11.300000,4.1,292.62,1.199742,0.030519
2,2021-03-28 00:00:22.600,2021-03-28,00:00:22.600000,4.1,351.19,1.439879,0.030519
3,2021-03-28 00:00:33.900,2021-03-28,00:00:33.900000,4.1,307.26,1.259766,0.042482
4,2021-03-28 00:00:45.200,2021-03-28,00:00:45.200000,4.1,257.48,1.055668,0.030512


In [11]:
df_birds.describe()

,measurement_datetime,time_offset_s,battery_voltage_V,battery_current_mA,battery_temperature_C,vpx_mV,vpy_mV,vpz_mV,vmx_mV,vmz_mV,...,ipz_mA,imx_mA,imz_mA,tpx_C,tpy_C,tpz_C,tmx_C,tmz_C,battery_power_W,solar_power_W
count,23266,23266.000000,23266.000000,23266.000000,23266.000000,23266.000000,23266.000000,23266.000000,23266.000000,23266.000000,...,23266.000000,23266.000000,23266.000000,23266.000000,23266.000000,23266.000000,23266.000000,23266.000000,23266.000000,23266.000000
mean,2021-08-24 09:23:32.512799,3149.536783,4.119310,25.616415,5.747117,3492.086917,3514.688620,3518.294199,3472.317988,3492.512777,...,34.866096,42.597991,42.051972,-0.448978,0.161983,2.443077,-1.716503,-3.192226,0.093401,1.079217
min,2021-03-28 00:00:00,0.000000,0.000000,-653.240000,-0.080000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,-30.210000,-28.220000,-32.210000,-26.660000,-32.210000,-2.724011,0.000000
25%,2021-06-11 00:43:22.500000,1470.000000,4.040000,-79.280000,2.600000,1263.740000,1320.210000,1275.950000,1311.050000,1275.950000,...,4.660000,0.000000,4.660000,-12.900000,-12.010000,-13.010000,-12.240000,-15.450000,-0.332976,0.036188
50%,2021-09-03 00:57:15,2980.000000,4.170000,-23.640000,4.900000,4653.540000,4659.650000,4674.910000,4642.860000,4648.960000,...,9.320000,0.000000,4.660000,0.190000,1.190000,1.970000,-3.020000,-5.580000,-0.099288,1.210313
75%,2021-11-05 00:44:17.500000,4570.000000,4.170000,236.980000,7.200000,5175.520000,5201.470000,5154.150000,5164.840000,5192.310000,...,27.970000,46.610000,46.610000,10.960000,11.730000,14.510000,7.960000,6.300000,0.952660,1.585363
max,2022-01-24 01:51:18.300000,12362.200000,4.200000,1200.420000,24.440000,5627.290000,5633.390000,5572.340000,5630.340000,5711.230000,...,484.750000,508.060000,470.770000,40.810000,45.360000,54.570000,41.030000,50.350000,4.585604,4.378088
std,NaN,2091.914333,0.110342,206.154064,4.764559,1848.407719,1845.777547,1826.139815,1840.373844,1868.700123,...,63.347800,80.597417,69.534503,14.702409,14.241966,18.857467,13.168604,16.441587,0.838633,0.907367


## 5 bis. Création des indicateurs énergétiques

À partir des mesures électriques réelles du CubeSat TSURU, on calcule plusieurs indicateurs énergétiques :

- la puissance produite par les panneaux solaires ;
- la puissance batterie ;
- un niveau de batterie estimé à partir de la tension ;
- une consommation estimée ;
- une balance énergétique.

Le niveau de batterie est estimé par normalisation de la tension batterie entre sa valeur minimale et sa valeur maximale observée dans le dataset.

In [12]:
# Estimation du niveau de batterie à partir de la tension observée
vbat_min = df_birds["battery_voltage_V"].quantile(0.01)
vbat_max = df_birds["battery_voltage_V"].quantile(0.99)

df_birds["battery_level_percent"] = (
    (df_birds["battery_voltage_V"] - vbat_min)
    / (vbat_max - vbat_min)
    * 100
)

df_birds["battery_level_percent"] = df_birds["battery_level_percent"].clip(0, 100)

In [13]:
# Consommation estimée
# Si la batterie se décharge, on ajoute la puissance fournie par la batterie à la consommation.
# Si la batterie se charge, la consommation est approximée par la production solaire moins la puissance stockée.
df_birds["consumption_W"] = np.where(
    df_birds["battery_power_W"] < 0,
    df_birds["solar_power_W"] + abs(df_birds["battery_power_W"]),
    df_birds["solar_power_W"] - df_birds["battery_power_W"]
)

df_birds["consumption_W"] = df_birds["consumption_W"].clip(lower=0)

In [14]:
# Balance énergétique
df_birds["energy_balance_W"] = (
    df_birds["solar_power_W"]
    - df_birds["consumption_W"]
)

In [15]:
df_birds[
    [
        "measurement_datetime",
        "battery_voltage_V",
        "battery_current_mA",
        "battery_level_percent",
        "solar_power_W",
        "battery_power_W",
        "consumption_W",
        "energy_balance_W"
    ]
].head()

,measurement_datetime,battery_voltage_V,battery_current_mA,battery_level_percent,solar_power_W,battery_power_W,consumption_W,energy_balance_W
0,2021-03-28 00:00:00.000,4.1,280.91,64.285714,0.036493,1.151731,0.0,0.036493
1,2021-03-28 00:00:11.300,4.1,292.62,64.285714,0.030519,1.199742,0.0,0.030519
2,2021-03-28 00:00:22.600,4.1,351.19,64.285714,0.030519,1.439879,0.0,0.030519
3,2021-03-28 00:00:33.900,4.1,307.26,64.285714,0.042482,1.259766,0.0,0.042482
4,2021-03-28 00:00:45.200,4.1,257.48,64.285714,0.030512,1.055668,0.0,0.030512


## 6. Récupération de la météo spatiale GFZ

On récupère les indices de météo spatiale correspondant à la période couverte par TSURU :

- Kp : activité géomagnétique ;
- ap : indice géomagnétique ;
- SN : nombre de taches solaires ;
- Fobs : flux solaire observé.

In [16]:
def classify_geomagnetic_level(kp):
    if kp < 2:
        return "quiet"
    elif kp < 4:
        return "unsettled"
    elif kp < 5:
        return "active"
    elif kp < 6:
        return "minor_storm"
    elif kp < 7:
        return "moderate_storm"
    elif kp < 8:
        return "strong_storm"
    elif kp < 9:
        return "severe_storm"
    else:
        return "extreme_storm"

In [17]:
def fetch_gfz_index(index_name, output_col, start, end):
    url = (
        "https://kp.gfz.de/app/json/"
        f"?start={start}"
        f"&end={end}"
        f"&index={index_name}"
    )

    response = rq.get(url)
    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame({
        "time_tag": pd.to_datetime(data["datetime"], utc=True).tz_convert(None),
        output_col: data[index_name]
    })

    return df

In [18]:
start_date = df_birds["measurement_datetime"].min().strftime("%Y-%m-%dT00:00:00Z")
end_date = df_birds["measurement_datetime"].max().strftime("%Y-%m-%dT23:59:59Z")

start_date, end_date

('2021-03-28T00:00:00Z', '2022-01-24T23:59:59Z')

In [19]:
df_kp = fetch_gfz_index("Kp", "kp_index", start_date, end_date)
df_ap = fetch_gfz_index("ap", "ap", start_date, end_date)
df_sn = fetch_gfz_index("SN", "sn", start_date, end_date)
df_fobs = fetch_gfz_index("Fobs", "fobs", start_date, end_date)

## 7. Fusion des indices de météo spatiale

Les indices n'ont pas tous la même fréquence.

On utilise `merge_asof` afin d'associer chaque valeur à la dernière mesure disponible.

In [20]:
df_weather = df_kp.sort_values("time_tag")

for df_idx in [df_ap, df_sn, df_fobs]:
    df_weather = pd.merge_asof(
        df_weather.sort_values("time_tag"),
        df_idx.sort_values("time_tag"),
        on="time_tag",
        direction="backward"
    )

df_weather["geomagnetic_level"] = df_weather["kp_index"].apply(
    classify_geomagnetic_level
)

df_weather.head()

,time_tag,kp_index,ap,sn,fobs,geomagnetic_level
0,2021-03-28 00:00:00,3.333,18.0,12.0,73.9,unsettled
1,2021-03-28 03:00:00,0.667,3.0,12.0,73.9,quiet
2,2021-03-28 06:00:00,1.000,4.0,12.0,73.9,quiet
3,2021-03-28 09:00:00,0.333,2.0,12.0,73.9,quiet
4,2021-03-28 12:00:00,0.667,3.0,12.0,73.9,quiet


In [21]:
df_weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 2424 entries, 0 to 2423
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   time_tag           2424 non-null   datetime64[us]
 1   kp_index           2424 non-null   float64       
 2   ap                 2424 non-null   float64       
 3   sn                 2424 non-null   float64       
 4   fobs               2424 non-null   float64       
 5   geomagnetic_level  2424 non-null   str           
dtypes: datetime64[us](1), float64(4), str(1)
memory usage: 113.8 KB


# Enrichissement orbital ISS

Afin d'enrichir les données énergétiques du CubeSat TSURU, des paramètres orbitaux de l'ISS sont ajoutés à chaque mesure.

Le satellite TSURU ayant été déployé depuis l'ISS sur une orbite basse (~400 km, inclinaison 51,6°), les paramètres orbitaux de l'ISS constituent une approximation réaliste de l'environnement orbital du CubeSat.

Les positions sont calculées à partir des éléments orbitaux TLE (Two Line Elements) de l'ISS grâce à la bibliothèque Skyfield.

## Chargement des bibliothèques orbitales

In [31]:
from skyfield.api import load, EarthSatellite
import requests as rq

## Récupération des éléments orbitaux de l'ISS

Les TLE (Two Line Elements) décrivent l'orbite du satellite et permettent de calculer sa position à n'importe quel instant.

In [32]:
API_tle_ISS = "https://celestrak.org/NORAD/elements/gp.php?CATNR=25544&FORMAT=tle"

response = rq.get(API_tle_ISS)

tle_text = response.text.strip().splitlines()

name = tle_text[0]
line1 = tle_text[1]
line2 = tle_text[2]

print(name)
print(line1)
print(line2)

ISS (ZARYA)             
1 25544U 98067A   26171.15661001  .00008669  00000+0  16345-3 0  9997
2 25544  51.6329 285.3962 0004555 207.7610 152.3136 15.49327998572210


## Création du modèle orbital ISS

In [33]:
ts = load.timescale()

satellite = EarthSatellite(
    line1=line1,
    line2=line2,
    name=name,
    ts=ts
)

## Préparation des instants de calcul

Les positions orbitales seront calculées pour chaque mesure de télémétrie disponible dans le dataset BIRDS.

In [34]:
timestamps = pd.to_datetime(
    df_birds["measurement_datetime"],
    utc=True
)

## Calcul des paramètres orbitaux

Pour chaque mesure :

- latitude
- longitude
- altitude

sont calculées à partir du modèle orbital de l'ISS.

In [35]:
orbit_rows = []

for dt in timestamps:

    t = ts.from_datetime(dt.to_pydatetime())

    geocentric = satellite.at(t)

    subpoint = geocentric.subpoint()

    orbit_rows.append({
        "measurement_datetime": dt,
        "latitude": subpoint.latitude.degrees,
        "longitude": subpoint.longitude.degrees,
        "altitude_km": subpoint.elevation.km
    })

df_orbit = pd.DataFrame(orbit_rows)

df_orbit.head()

,measurement_datetime,latitude,longitude,altitude_km
0,2021-03-28 00:00:00+00:00,-47.623622,-169.718087,470.855164
1,2021-03-28 00:00:11.300000+00:00,-47.905790,-168.778485,470.903206
2,2021-03-28 00:00:22.600000+00:00,-48.179387,-167.828274,470.948018
3,2021-03-28 00:00:33.900000+00:00,-48.444242,-166.867575,470.989550
4,2021-03-28 00:00:45.200000+00:00,-48.700181,-165.896528,471.027759


## Vérification rapide des données orbitales

In [36]:
print(df_orbit.describe())

           latitude     longitude   altitude_km
count  23266.000000  23266.000000  23266.000000
mean      -0.235723     -2.106761    468.456500
std       35.026378    104.167957      7.155732
min      -51.787479   -179.993941    452.879741
25%      -33.693907    -90.884965    463.167732
50%       -0.336581     -4.837981    468.106698
75%       33.179049     87.709363    473.053209
max       51.787790    179.999634    487.712869


## Fusion des paramètres orbitaux avec la télémétrie énergétique

In [38]:
df_orbit["measurement_datetime"] = (
    df_orbit["measurement_datetime"]
    .dt.tz_localize(None)
)

In [39]:
df_orbit["measurement_datetime"] = pd.to_datetime(
    df_orbit["measurement_datetime"]
)

df_birds["measurement_datetime"] = pd.to_datetime(
    df_birds["measurement_datetime"]
)

df_birds = df_birds.merge(
    df_orbit,
    on="measurement_datetime",
    how="left"
)

## Vérification des nouvelles colonnes

In [40]:
df_birds[
    [
        "measurement_datetime",
        "latitude",
        "longitude",
        "altitude_km"
    ]
].head()

,measurement_datetime,latitude,longitude,altitude_km
0,2021-03-28 00:00:00.000,-47.623622,-169.718087,470.855164
1,2021-03-28 00:00:11.300,-47.905790,-168.778485,470.903206
2,2021-03-28 00:00:22.600,-48.179387,-167.828274,470.948018
3,2021-03-28 00:00:33.900,-48.444242,-166.867575,470.989550
4,2021-03-28 00:00:45.200,-48.700181,-165.896528,471.027759


## Vérification des valeurs manquantes

In [41]:
df_birds[
    [
        "latitude",
        "longitude",
        "altitude_km"
    ]
].isna().sum()

latitude       0
longitude      0
altitude_km    0
dtype: int64

## 8. Export CSV de sécurité

Ces fichiers peuvent être utilisés comme sauvegarde ou comme sources alternatives pour Power BI / SQL Server.

In [42]:
df_birds.to_csv(
    "birds_tsuru_clean.csv",
    index=False,
    sep=";",
    decimal=","
)

df_weather.to_csv(
    "birds_space_weather.csv",
    index=False,
    sep=";",
    decimal=","
)

## 9. Envoi des données vers SQL Server

Les données sont envoyées dans les tables de staging :

- `BIRDS_Stg_eps_telemetry`
- `BIRDS_Stg_weather`

La base utilisée est la même que le projet principal : `SatelliteDW`.

In [43]:
from sqlalchemy import create_engine
import urllib

In [45]:
SERVER = "localhost"
DATABASE = "SatelliteDW"

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

## 10. Vidage des tables de staging

In [47]:
with engine.begin() as conn:
    #conn.exec_driver_sql("DELETE FROM BIRDS_Stg_eps_telemetry;")
    conn.exec_driver_sql("DELETE FROM BIRDS_Stg_weather;")

## 11. Chargement des données dans SQL Server

In [48]:
df_birds.to_sql(
    "BIRDS_Stg_eps_telemetry",
    engine,
    if_exists="append",
    index=False
)

df_weather.to_sql(
    "BIRDS_Stg_weather",
    engine,
    if_exists="append",
    index=False
)

print("Import vers SQL Server terminé.")

Import vers SQL Server terminé.


## 12. Vérification finale

In [49]:
print("Nombre de lignes TSURU :", len(df_birds))
print("Nombre de lignes météo spatiale :", len(df_weather))

print("Période TSURU :")
print(df_birds["measurement_datetime"].min())
print(df_birds["measurement_datetime"].max())

print("Période météo spatiale :")
print(df_weather["time_tag"].min())
print(df_weather["time_tag"].max())

Nombre de lignes TSURU : 23266
Nombre de lignes météo spatiale : 2424
Période TSURU :
2021-03-28 00:00:00
2022-01-24 01:51:18.300000
Période météo spatiale :
2021-03-28 00:00:00
2022-01-24 21:00:00
